In [3]:
import shutil
import os
import glob
import sys

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

sys.path.append("..")
from utils import data_reading


load_dotenv("../.env")

True

In [4]:
CVAT_USER = os.getenv("CVAT_USER")
CVAT_PASSWORD = os.getenv("CVAT_PASSWORD")

BASE_PATH = "/home/klymy/data/ObjectDetectionV3_2_dbg"
INPUT_PATH = os.path.join(BASE_PATH, "annotation.zip")

ANNO_PATH = os.path.join(BASE_PATH, "annotation")

In [5]:
shutil.unpack_archive(INPUT_PATH, BASE_PATH)

In [6]:
batches = sorted(glob.glob(os.path.join(ANNO_PATH, '*')))

subbatches = []
for batch in batches:
    subbatches += sorted(glob.glob(os.path.join(batch, "*.zip")))

for subbatch in tqdm(subbatches):
    shutil.unpack_archive(subbatch, subbatch.replace(".zip", ""))
    os.remove(subbatch)


subbatches = [subbatch.replace(".zip", "") for subbatch in subbatches]

100%|██████████| 67/67 [00:02<00:00, 29.62it/s]


In [22]:
imgs = {}
annotations = {}
total_img_count = 0
for batch in subbatches:
    batch_imgs = [i for i in os.listdir(os.path.join(batch, "images")) if data_reading.is_img(i)]
    batch_annotation = [i for i in os.listdir(batch) if i.endswith(".xml")][0]
    
    imgs[batch] = batch_imgs
    annotations[batch] = batch_annotation
    total_img_count += len(batch_imgs)

print("total images:", total_img_count)

total images: 2065


In [41]:
objects = []
img_attributes = []

for batch, annot in tqdm(annotations.items()):
    tmp_objects, tmp_img_attributes = data_reading.process_annotation_file(
        os.path.join(batch, annot),
        CVAT_USER,
        CVAT_PASSWORD,
        *batch.split("/")[-2:])
    objects += tmp_objects
    img_attributes += tmp_img_attributes

100%|██████████| 1/1 [01:41<00:00, 101.85s/it]


In [29]:
df = pd.DataFrame(objects)
df_imgs = pd.DataFrame(img_attributes)
print(df_imgs.shape)

for batch, batch_imgs in imgs.items():
    subbatch, folder = batch.split("/")[-2:]
    for img in batch_imgs:
        if ((df_imgs["name"] == img) & (df_imgs["batch"] == subbatch) & (df_imgs["folder"] == folder)).any():
            continue

        df_imgs = pd.concat([df_imgs, pd.DataFrame([{"id": data_reading.gen_uuid(), "name": img, "batch": subbatch, "folder": folder}])], ignore_index=True)

(50, 9)


In [31]:
print("total number of images: ", df_imgs.shape[0])

print("total number of uniques images with annotations: ", df["img_id"].unique().shape[0])

print("total number of annotated objects: ", df.shape[0])

object_per_img = df.groupby("img_id").count()["id"].values
print(f"objects per img: min = {object_per_img.min()}. max = {object_per_img.max()}. avg = {object_per_img.mean():.02f}")

print("number of object per class:\n", df["label"].value_counts())

total number of images:  2065
total number of uniques images with annotations:  50
total number of annotated objects:  1524
objects per img: min = 17. max = 41. avg = 30.48
number of object per class:
 label
office_chair               570
laptop_opened              234
desk                       106
monitor                     98
desk_FULL                   98
keyboard                    78
plants                      54
sofa                        45
meeting_room_table          43
lounge_chair                41
cabinet_FULL                38
coffee_table_FULL           31
coffee_table                30
cabinet                     27
meeting_room_table_FULL     13
coat                        13
dinning_chair                2
bag                          2
laptop_closed                1
Name: count, dtype: int64


In [33]:
tmp_df = df[(df["xtl"].isna()) | (df["x"].isna())]
tmp_df = pd.merge(tmp_df, df_imgs, how="left", left_on="img_id", right_on="id")[["job_url", "batch", "folder", "name", "label", "ID", "xtl", "ytl", "xbr", "ybr", "x", "y"]]
tmp_df.to_csv(os.path.join(BASE_PATH, "no_boxes_or_position.csv"), index=False)
print("images with no boxes or position:", tmp_df["name"].unique().shape[0])

tmp_df = pd.merge(df, df_imgs, how="outer", left_on="img_id", right_on="id")
tmp_df = tmp_df[tmp_df["id_x"].isna()][["job_url", "batch", "folder","name"]].sort_values(by=["batch", "folder"])
tmp_df.to_csv(os.path.join(BASE_PATH, "no_annotations.csv"), index=False)
print("images with no annotations:", tmp_df["name"].unique().shape[0])

df1 = df_imgs.groupby(['name','batch', "folder"]).size().reset_index()[['name','batch', "folder"]]
duplicates = df1[df1.duplicated('name', keep=False)]
duplicate_info = duplicates.groupby('name').agg({
    'folder': lambda x: ', '.join(x),
    'batch': lambda x: ', '.join(x)
}).reset_index()
duplicate_info.to_csv(os.path.join(BASE_PATH, "duplicates.csv"), index=False)
print("duplicates:", duplicate_info.shape[0])

images with no boxes or position: 1
images with no annotations: 2015
duplicates: 0


In [17]:
df.to_csv(os.path.join(BASE_PATH, "annotation.csv"), index=False)
df_imgs.to_csv(os.path.join(BASE_PATH, "images.csv"), index=False)